# Model Evaluation and Analysis

This notebook provides comprehensive evaluation and analysis of the trained Lexical Semantic Embedding Model. We'll evaluate performance across multiple datasets, analyze results, and create visualizations.

## Table of Contents
1. [Setup and Model Loading](#Setup-and-Model-Loading)
2. [Dataset Evaluation](#Dataset-Evaluation)
3. [Performance Metrics](#Performance-Metrics)
4. [Error Analysis](#Error-Analysis)
5. [Embedding Visualization](#Embedding-Visualization)
6. [Semantic Search Demo](#Semantic-Search-Demo)
7. [Model Comparison](#Model-Comparison)
8. [Results Export](#Results-Export)

In [1]:
# Setup environment
import sys
import os
from pathlib import Path

# Add project root to path
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

# Import required libraries
import torch
import torch.nn.functional as F
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import json
import pickle
from datetime import datetime

# Import project modules
from lexical_embedding_model import LexicalSemanticEmbeddingModel
from config.model_config import ModelConfig
from config.data_config import DataConfig
from scripts.evaluate import ModelEvaluator
from scripts.utils import setup_logging, get_device, compute_correlation_metrics

# Setup
setup_logging("INFO")
device = get_device()
print(f"Using device: {device}")
print(f"Project root: {PROJECT_ROOT}")

2025-09-19 11:54:55,424 - scripts.utils - INFO - Logging setup complete. Level: INFO, File: None
2025-09-19 11:54:55,425 - scripts.utils - INFO - Using CPU
2025-09-19 11:54:55,425 - scripts.utils - INFO - Using CPU


Using device: cpu
Project root: /home/aditya/Documents/Github/Lexical_Model/lexical_embedding_project


## Model Loading and Configuration

In [2]:
# Model and data configuration
model_config = ModelConfig.get_config("medium")
data_config = DataConfig()

# Load trained model (or create for demo)
MODEL_PATH = PROJECT_ROOT / "models" / "best_model.pt"

if MODEL_PATH.exists():
    print(f"Loading model from {MODEL_PATH}")
    model = LexicalSemanticEmbeddingModel.load_model(MODEL_PATH, device)
else:
    print("Creating new model for demonstration")
    model = LexicalSemanticEmbeddingModel(model_config)
    model = model.to(device)

# Model info
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Model loaded successfully!")
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Model config: {model_config.model_name}")

AttributeError: type object 'ModelConfig' has no attribute 'get_config'

In [ ]:
# Create evaluator
evaluator = ModelEvaluator(
    model=model,
    data_config=data_config,
    device=device,
    batch_size=32
)

print("Model evaluator created successfully!")

## Sample Data Generation for Evaluation Demo

In [ ]:
# Generate sample evaluation data
np.random.seed(42)

# High similarity pairs
high_sim_pairs = [
    ("The cat sat on the mat.", "A cat was sitting on the mat.", 4.5),
    ("He is playing football.", "He is playing soccer.", 4.2),
    ("The dog is running fast.", "A dog runs quickly.", 4.0),
    ("She loves reading books.", "She enjoys reading novels.", 3.8),
    ("The car is red in color.", "The automobile is red.", 4.1),
    ("Birds can fly in the sky.", "Birds are able to fly.", 3.9),
    ("The weather is nice today.", "Today has beautiful weather.", 4.3),
    ("I am going to the store.", "I'm heading to the shop.", 3.7),
    ("The movie was very interesting.", "The film was quite engaging.", 4.0),
    ("Water is essential for life.", "Water is necessary for living.", 4.4)
]

# Medium similarity pairs
medium_sim_pairs = [
    ("The sun is shining brightly.", "It's a beautiful day outside.", 2.8),
    ("I need to buy groceries.", "I have to go shopping.", 3.2),
    ("The book is on the table.", "There's reading material nearby.", 2.5),
    ("He drives to work daily.", "His commute involves transportation.", 2.9),
    ("The flowers smell wonderful.", "The garden has a pleasant aroma.", 3.1),
    ("She studies mathematics.", "She's learning about numbers.", 2.7),
    ("The phone is ringing.", "Someone is trying to call.", 3.0),
    ("Dinner is ready now.", "The meal has been prepared.", 3.3),
    ("The train arrives soon.", "Transportation will be here shortly.", 2.6),
    ("Music plays softly.", "There are gentle sounds around.", 2.4)
]

# Low similarity pairs
low_sim_pairs = [
    ("The cat sat on the mat.", "Quantum physics is complex.", 0.1),
    ("I love eating pizza.", "The stock market is volatile.", 0.2),
    ("Birds can fly high.", "Mathematics requires practice.", 0.1),
    ("The ocean is deep.", "Programming languages vary.", 0.3),
    ("Music sounds beautiful.", "Economic theories differ.", 0.2),
    ("Children play games.", "Molecular biology advances.", 0.1),
    ("Flowers bloom spring.", "Political systems evolve.", 0.2),
    ("Cars need fuel.", "Archaeological discoveries continue.", 0.1),
    ("Rain falls down.", "Artificial intelligence progresses.", 0.3),
    ("Books contain stories.", "Chemical reactions occur.", 0.2)
]

# Combine all pairs
all_pairs = high_sim_pairs + medium_sim_pairs + low_sim_pairs

# Create evaluation dataset
eval_data = {
    'sentence1': [pair[0] for pair in all_pairs],
    'sentence2': [pair[1] for pair in all_pairs],
    'true_score': [pair[2] for pair in all_pairs]
}

eval_df = pd.DataFrame(eval_data)
print(f"Created evaluation dataset with {len(eval_df)} sentence pairs")
print(f"Score range: {eval_df['true_score'].min():.1f} - {eval_df['true_score'].max():.1f}")

## Model Evaluation and Metrics

In [ ]:
# Run evaluation
print("Running model evaluation...")

# Evaluate on sample data
evaluation_results = evaluator.evaluate_pairs(
    sentences1=eval_df['sentence1'].tolist(),
    sentences2=eval_df['sentence2'].tolist(),
    true_scores=eval_df['true_score'].tolist(),
    dataset_name="sample_evaluation"
)

print("\nEvaluation Results:")
print("=" * 40)
for metric, value in evaluation_results.items():
    if isinstance(value, (int, float)):
        print(f"{metric}: {value:.4f}")
    else:
        print(f"{metric}: {value}")

In [ ]:
# Get model predictions for detailed analysis
model.eval()
predictions = []

print("Getting model predictions...")

with torch.no_grad():
    for i, row in eval_df.iterrows():
        # Simple tokenization (in practice, use proper tokenizer)
        tokens1 = row['sentence1'].lower().split()[:50]
        tokens2 = row['sentence2'].lower().split()[:50]
        
        # Convert to IDs (simplified)
        ids1 = [hash(token) % 30000 + 1 for token in tokens1]
        ids2 = [hash(token) % 30000 + 1 for token in tokens2]
        
        # Pad sequences
        max_len = 50
        ids1 += [0] * (max_len - len(ids1))
        ids2 += [0] * (max_len - len(ids2))
        
        # Create attention masks
        mask1 = [1 if id > 0 else 0 for id in ids1]
        mask2 = [1 if id > 0 else 0 for id in ids2]
        
        # Convert to tensors
        input_ids_1 = torch.tensor([ids1], device=device)
        input_ids_2 = torch.tensor([ids2], device=device)
        attention_mask_1 = torch.tensor([mask1], device=device)
        attention_mask_2 = torch.tensor([mask2], device=device)
        
        # Forward pass
        output = model(
            input_ids_1=input_ids_1,
            input_ids_2=input_ids_2,
            attention_mask_1=attention_mask_1,
            attention_mask_2=attention_mask_2
        )
        
        pred_score = output['similarity'].item()
        predictions.append(pred_score)

# Add predictions to dataframe
eval_df['predicted_score'] = predictions
eval_df['error'] = eval_df['predicted_score'] - eval_df['true_score']
eval_df['abs_error'] = abs(eval_df['error'])

print(f"Predictions completed for {len(predictions)} pairs")
print(f"Mean Absolute Error: {eval_df['abs_error'].mean():.4f}")
print(f"RMSE: {np.sqrt((eval_df['error']**2).mean()):.4f}")

## Performance Visualization

In [ ]:
# Create prediction vs actual scatter plot
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# 1. Predictions vs Actual
ax1 = axes[0, 0]
ax1.scatter(eval_df['true_score'], eval_df['predicted_score'], alpha=0.7, s=50)
ax1.plot([0, 5], [0, 5], 'r--', label='Perfect Prediction')
ax1.set_xlabel('True Score')
ax1.set_ylabel('Predicted Score')
ax1.set_title('Predictions vs Actual Scores')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Calculate R²
correlation = np.corrcoef(eval_df['true_score'], eval_df['predicted_score'])[0, 1]
r_squared = correlation ** 2
ax1.text(0.05, 0.95, f'R² = {r_squared:.3f}', transform=ax1.transAxes, 
         bbox=dict(boxstyle='round', facecolor='wheat'))

# 2. Error distribution
ax2 = axes[0, 1]
ax2.hist(eval_df['error'], bins=15, alpha=0.7, edgecolor='black')
ax2.axvline(0, color='red', linestyle='--', label='Zero Error')
ax2.set_xlabel('Prediction Error')
ax2.set_ylabel('Frequency')
ax2.set_title('Error Distribution')
ax2.legend()
ax2.grid(True, alpha=0.3)

# 3. Absolute error by true score
ax3 = axes[1, 0]
ax3.scatter(eval_df['true_score'], eval_df['abs_error'], alpha=0.7, s=50)
ax3.set_xlabel('True Score')
ax3.set_ylabel('Absolute Error')
ax3.set_title('Absolute Error by True Score')
ax3.grid(True, alpha=0.3)

# 4. Score distribution comparison
ax4 = axes[1, 1]
ax4.hist(eval_df['true_score'], alpha=0.5, label='True Scores', bins=15)
ax4.hist(eval_df['predicted_score'], alpha=0.5, label='Predicted Scores', bins=15)
ax4.set_xlabel('Score')
ax4.set_ylabel('Frequency')
ax4.set_title('Score Distribution Comparison')
ax4.legend()
ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Interactive analysis with Plotly
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=[
        "Predictions vs Actual (Interactive)",
        "Error Distribution",
        "Absolute Error by Score Range",
        "Model Confidence Analysis"
    ]
)

# Add hover text with sentence pairs
hover_text = [
    f"Sent1: {row['sentence1'][:50]}...<br>Sent2: {row['sentence2'][:50]}...<br>True: {row['true_score']:.2f}<br>Pred: {row['predicted_score']:.2f}"
    for _, row in eval_df.iterrows()
]

# 1. Interactive scatter plot
fig.add_trace(
    go.Scatter(
        x=eval_df['true_score'],
        y=eval_df['predicted_score'],
        mode='markers',
        text=hover_text,
        hovertemplate='%{text}<extra></extra>',
        marker=dict(size=8, opacity=0.7, color=eval_df['abs_error'], 
                   colorscale='Viridis', showscale=True,
                   colorbar=dict(title="Abs Error"))
    ),
    row=1, col=1
)

# Add perfect prediction line
fig.add_trace(
    go.Scatter(
        x=[0, 5], y=[0, 5],
        mode='lines',
        line=dict(color='red', dash='dash'),
        name='Perfect Prediction'
    ),
    row=1, col=1
)

# 2. Error histogram
fig.add_trace(
    go.Histogram(
        x=eval_df['error'],
        nbinsx=15,
        name='Error Distribution'
    ),
    row=1, col=2
)

# 3. Boxplot of errors by score ranges
eval_df['score_range'] = pd.cut(eval_df['true_score'], bins=3, labels=['Low', 'Medium', 'High'])
for range_name in ['Low', 'Medium', 'High']:
    range_data = eval_df[eval_df['score_range'] == range_name]
    fig.add_trace(
        go.Box(
            y=range_data['abs_error'],
            name=range_name,
            boxpoints='all',
            jitter=0.3,
            pointpos=-1.8
        ),
        row=2, col=1
    )

# 4. Confidence analysis (prediction variance by score)
score_bins = np.linspace(0, 5, 11)
bin_centers = (score_bins[:-1] + score_bins[1:]) / 2
bin_errors = []

for i in range(len(score_bins)-1):
    mask = (eval_df['true_score'] >= score_bins[i]) & (eval_df['true_score'] < score_bins[i+1])
    if mask.sum() > 0:
        bin_errors.append(eval_df[mask]['abs_error'].mean())
    else:
        bin_errors.append(0)

fig.add_trace(
    go.Bar(
        x=bin_centers,
        y=bin_errors,
        name='Avg Error by Score Range'
    ),
    row=2, col=2
)

fig.update_layout(
    title_text="Comprehensive Model Performance Analysis",
    height=800,
    showlegend=False
)

fig.show()

## Error Analysis and Edge Cases

In [ ]:
# Analyze worst predictions
worst_predictions = eval_df.nlargest(10, 'abs_error')
best_predictions = eval_df.nsmallest(10, 'abs_error')

print("WORST PREDICTIONS (Highest Absolute Error):")
print("=" * 60)
for idx, row in worst_predictions.iterrows():
    print(f"\nError: {row['error']:.3f} (|{row['abs_error']:.3f}|)")
    print(f"True: {row['true_score']:.2f}, Predicted: {row['predicted_score']:.2f}")
    print(f"Sent1: {row['sentence1']}")
    print(f"Sent2: {row['sentence2']}")
    print("-" * 40)

print("\n\nBEST PREDICTIONS (Lowest Absolute Error):")
print("=" * 60)
for idx, row in best_predictions.iterrows():
    print(f"\nError: {row['error']:.3f} (|{row['abs_error']:.3f}|)")
    print(f"True: {row['true_score']:.2f}, Predicted: {row['predicted_score']:.2f}")
    print(f"Sent1: {row['sentence1']}")
    print(f"Sent2: {row['sentence2']}")
    print("-" * 40)

In [ ]:
# Analyze error patterns
eval_df['sentence1_length'] = eval_df['sentence1'].str.len()
eval_df['sentence2_length'] = eval_df['sentence2'].str.len()
eval_df['length_diff'] = abs(eval_df['sentence1_length'] - eval_df['sentence2_length'])
eval_df['avg_length'] = (eval_df['sentence1_length'] + eval_df['sentence2_length']) / 2

# Correlation analysis
correlations = {
    'Length vs Error': eval_df['avg_length'].corr(eval_df['abs_error']),
    'Length Diff vs Error': eval_df['length_diff'].corr(eval_df['abs_error']),
    'True Score vs Error': eval_df['true_score'].corr(eval_df['abs_error'])
}

print("ERROR PATTERN ANALYSIS:")
print("=" * 40)
for pattern, corr in correlations.items():
    print(f"{pattern}: {corr:.4f}")

# Error by score range
score_ranges = {
    'Low (0-1.5)': eval_df[eval_df['true_score'] <= 1.5],
    'Medium (1.5-3.5)': eval_df[(eval_df['true_score'] > 1.5) & (eval_df['true_score'] <= 3.5)],
    'High (3.5-5)': eval_df[eval_df['true_score'] > 3.5]
}

print("\nERROR BY SCORE RANGE:")
print("=" * 40)
for range_name, range_df in score_ranges.items():
    if len(range_df) > 0:
        avg_error = range_df['abs_error'].mean()
        std_error = range_df['abs_error'].std()
        print(f"{range_name}: Avg Error = {avg_error:.4f} ± {std_error:.4f} ({len(range_df)} samples)")
    else:
        print(f"{range_name}: No samples")

## Semantic Search Demonstration

In [ ]:
# Semantic search demonstration
candidate_sentences = [
    "The cat sat on the comfortable mat.",
    "A dog is running quickly through the park.",
    "She loves reading interesting books every evening.",
    "The weather today is absolutely beautiful.",
    "Mathematics can be quite challenging to learn.",
    "Birds are flying high in the clear sky.",
    "The red car is parked outside.",
    "Music sounds wonderful when played softly.",
    "Cooking dinner takes time and patience.",
    "Technology continues to advance rapidly.",
    "The ocean waves crash against the shore.",
    "Children enjoy playing games together.",
    "Fresh flowers smell absolutely delightful.",
    "Learning new languages opens many doors.",
    "The sun sets beautifully over the horizon."
]

# Test queries
test_queries = [
    "A feline is resting on a rug.",
    "The climate is pleasant today.",
    "She enjoys literature.",
    "Canines run in recreational areas."
]

print("SEMANTIC SEARCH DEMONSTRATION:")
print("=" * 50)

for query in test_queries:
    print(f"\nQuery: '{query}'")
    print("-" * 30)
    
    # Get similarity scores
    similarities = evaluator.compute_embedding_similarities(candidate_sentences, query)
    
    # Display top 3 results
    for i, (sentence, score) in enumerate(similarities[:3], 1):
        print(f"{i}. [Score: {score:.4f}] {sentence}")
    
    print()

In [ ]:
# Visualize embeddings in 2D space
print("Creating embedding visualization...")

# Combine query and candidate sentences
all_sentences = test_queries + candidate_sentences[:10]  # Limit for visualization
labels = ['Query'] * len(test_queries) + ['Candidate'] * 10

# Analyze embeddings using PCA
embedding_analysis = evaluator.analyze_embeddings(
    sentences=all_sentences,
    labels=labels,
    method="pca"
)

if embedding_analysis:
    coordinates = embedding_analysis['coordinates']
    
    # Create interactive plot
    fig = go.Figure()
    
    # Plot queries
    query_coords = coordinates[:len(test_queries)]
    fig.add_trace(
        go.Scatter(
            x=query_coords[:, 0],
            y=query_coords[:, 1],
            mode='markers+text',
            marker=dict(size=12, color='red', symbol='star'),
            text=[f"Q{i+1}" for i in range(len(test_queries))],
            textposition="top center",
            name='Queries',
            hovertext=[f"Query {i+1}: {sent}" for i, sent in enumerate(test_queries)]
        )
    )
    
    # Plot candidates
    candidate_coords = coordinates[len(test_queries):]
    fig.add_trace(
        go.Scatter(
            x=candidate_coords[:, 0],
            y=candidate_coords[:, 1],
            mode='markers+text',
            marker=dict(size=8, color='blue', opacity=0.7),
            text=[f"C{i+1}" for i in range(len(candidate_coords))],
            textposition="top center",
            name='Candidates',
            hovertext=[f"Candidate {i+1}: {sent}" for i, sent in enumerate(candidate_sentences[:10])]
        )
    )
    
    fig.update_layout(
        title="Sentence Embeddings in 2D Space (PCA)",
        xaxis_title="First Principal Component",
        yaxis_title="Second Principal Component",
        width=800,
        height=600
    )
    
    fig.show()
    
    if 'variance_explained' in embedding_analysis and embedding_analysis['variance_explained'] is not None:
        print(f"Variance explained by first two components: {sum(embedding_analysis['variance_explained']):.3f}")
else:
    print("Embedding analysis not available (sklearn not installed)")

## Model Comparison and Benchmarking

In [ ]:
# Create simple baseline models for comparison
def simple_word_overlap_similarity(sent1, sent2):
    """Simple word overlap baseline."""
    words1 = set(sent1.lower().split())
    words2 = set(sent2.lower().split())
    
    intersection = len(words1 & words2)
    union = len(words1 | words2)
    
    if union == 0:
        return 0
    
    jaccard = intersection / union
    return jaccard * 5  # Scale to 0-5 range

def length_based_similarity(sent1, sent2):
    """Length-based similarity baseline."""
    len1, len2 = len(sent1), len(sent2)
    max_len = max(len1, len2)
    
    if max_len == 0:
        return 5
    
    similarity = 1 - abs(len1 - len2) / max_len
    return similarity * 5

# Calculate baseline predictions
eval_df['baseline_overlap'] = eval_df.apply(
    lambda row: simple_word_overlap_similarity(row['sentence1'], row['sentence2']), axis=1
)

eval_df['baseline_length'] = eval_df.apply(
    lambda row: length_based_similarity(row['sentence1'], row['sentence2']), axis=1
)

# Calculate metrics for all models
models_comparison = {
    'Our Model': {
        'predictions': eval_df['predicted_score'],
        'mae': eval_df['abs_error'].mean(),
        'rmse': np.sqrt((eval_df['error']**2).mean()),
        'pearson': eval_df['true_score'].corr(eval_df['predicted_score']),
        'spearman': eval_df['true_score'].corr(eval_df['predicted_score'], method='spearman')
    },
    'Word Overlap': {
        'predictions': eval_df['baseline_overlap'],
        'mae': abs(eval_df['baseline_overlap'] - eval_df['true_score']).mean(),
        'rmse': np.sqrt(((eval_df['baseline_overlap'] - eval_df['true_score'])**2).mean()),
        'pearson': eval_df['true_score'].corr(eval_df['baseline_overlap']),
        'spearman': eval_df['true_score'].corr(eval_df['baseline_overlap'], method='spearman')
    },
    'Length-based': {
        'predictions': eval_df['baseline_length'],
        'mae': abs(eval_df['baseline_length'] - eval_df['true_score']).mean(),
        'rmse': np.sqrt(((eval_df['baseline_length'] - eval_df['true_score'])**2).mean()),
        'pearson': eval_df['true_score'].corr(eval_df['baseline_length']),
        'spearman': eval_df['true_score'].corr(eval_df['baseline_length'], method='spearman')
    }
}

# Display comparison
comparison_df = pd.DataFrame({
    model: {metric: value for metric, value in metrics.items() if metric != 'predictions'}
    for model, metrics in models_comparison.items()
}).T

print("MODEL COMPARISON:")
print("=" * 50)
print(comparison_df.round(4))

In [ ]:
# Visualize model comparison
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# 1. Predictions comparison
ax1 = axes[0, 0]
ax1.scatter(eval_df['true_score'], eval_df['predicted_score'], alpha=0.7, label='Our Model', s=50)
ax1.scatter(eval_df['true_score'], eval_df['baseline_overlap'], alpha=0.7, label='Word Overlap', s=30)
ax1.scatter(eval_df['true_score'], eval_df['baseline_length'], alpha=0.7, label='Length-based', s=30)
ax1.plot([0, 5], [0, 5], 'r--', alpha=0.5)
ax1.set_xlabel('True Score')
ax1.set_ylabel('Predicted Score')
ax1.set_title('Model Predictions Comparison')
ax1.legend()
ax1.grid(True, alpha=0.3)

# 2. Metrics comparison
ax2 = axes[0, 1]
metrics_names = ['MAE', 'RMSE', 'Pearson', 'Spearman']
our_model_metrics = [comparison_df.loc['Our Model', 'mae'], 
                    comparison_df.loc['Our Model', 'rmse'],
                    comparison_df.loc['Our Model', 'pearson'],
                    comparison_df.loc['Our Model', 'spearman']]
overlap_metrics = [comparison_df.loc['Word Overlap', 'mae'],
                  comparison_df.loc['Word Overlap', 'rmse'],
                  comparison_df.loc['Word Overlap', 'pearson'],
                  comparison_df.loc['Word Overlap', 'spearman']]

x = np.arange(len(metrics_names))
width = 0.35

ax2.bar(x - width/2, our_model_metrics, width, label='Our Model', alpha=0.8)
ax2.bar(x + width/2, overlap_metrics, width, label='Word Overlap', alpha=0.8)
ax2.set_xlabel('Metrics')
ax2.set_ylabel('Score')
ax2.set_title('Metrics Comparison')
ax2.set_xticks(x)
ax2.set_xticklabels(metrics_names)
ax2.legend()
ax2.grid(True, alpha=0.3)

# 3. Error distribution comparison
ax3 = axes[1, 0]
our_errors = eval_df['predicted_score'] - eval_df['true_score']
overlap_errors = eval_df['baseline_overlap'] - eval_df['true_score']

ax3.hist(our_errors, alpha=0.6, label='Our Model', bins=15)
ax3.hist(overlap_errors, alpha=0.6, label='Word Overlap', bins=15)
ax3.axvline(0, color='red', linestyle='--', alpha=0.5)
ax3.set_xlabel('Prediction Error')
ax3.set_ylabel('Frequency')
ax3.set_title('Error Distribution Comparison')
ax3.legend()
ax3.grid(True, alpha=0.3)

# 4. Performance by score range
ax4 = axes[1, 1]
score_ranges = ['Low (0-1.5)', 'Medium (1.5-3.5)', 'High (3.5-5)']
our_range_errors = []
overlap_range_errors = []

for low, high in [(0, 1.5), (1.5, 3.5), (3.5, 5)]:
    mask = (eval_df['true_score'] >= low) & (eval_df['true_score'] <= high)
    if mask.sum() > 0:
        our_range_errors.append(abs(eval_df[mask]['predicted_score'] - eval_df[mask]['true_score']).mean())
        overlap_range_errors.append(abs(eval_df[mask]['baseline_overlap'] - eval_df[mask]['true_score']).mean())
    else:
        our_range_errors.append(0)
        overlap_range_errors.append(0)

x = np.arange(len(score_ranges))
ax4.bar(x - width/2, our_range_errors, width, label='Our Model', alpha=0.8)
ax4.bar(x + width/2, overlap_range_errors, width, label='Word Overlap', alpha=0.8)
ax4.set_xlabel('Score Range')
ax4.set_ylabel('Mean Absolute Error')
ax4.set_title('Error by Score Range')
ax4.set_xticks(x)
ax4.set_xticklabels(score_ranges, rotation=45)
ax4.legend()
ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Results Export and Summary

In [ ]:
# Prepare comprehensive results
results_summary = {
    'evaluation_timestamp': datetime.now().isoformat(),
    'model_info': {
        'total_parameters': total_params,
        'trainable_parameters': trainable_params,
        'model_config': model_config.model_name,
        'device': str(device)
    },
    'dataset_info': {
        'total_samples': len(eval_df),
        'score_range': f"{eval_df['true_score'].min():.1f} - {eval_df['true_score'].max():.1f}",
        'score_distribution': eval_df['true_score'].value_counts().to_dict()
    },
    'performance_metrics': {
        'mae': float(eval_df['abs_error'].mean()),
        'rmse': float(np.sqrt((eval_df['error']**2).mean())),
        'pearson_correlation': float(eval_df['true_score'].corr(eval_df['predicted_score'])),
        'spearman_correlation': float(eval_df['true_score'].corr(eval_df['predicted_score'], method='spearman')),
        'r_squared': float(eval_df['true_score'].corr(eval_df['predicted_score'])**2)
    },
    'comparison_with_baselines': comparison_df.to_dict(),
    'error_analysis': {
        'worst_predictions': worst_predictions[['sentence1', 'sentence2', 'true_score', 'predicted_score', 'abs_error']].to_dict('records'),
        'best_predictions': best_predictions[['sentence1', 'sentence2', 'true_score', 'predicted_score', 'abs_error']].to_dict('records')
    }
}

# Create results directory
results_dir = PROJECT_ROOT / "evaluation_results"
results_dir.mkdir(exist_ok=True)

# Save results
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
results_file = results_dir / f"evaluation_results_{timestamp}.json"

with open(results_file, 'w') as f:
    json.dump(results_summary, f, indent=2, default=str)

# Save detailed predictions
predictions_file = results_dir / f"detailed_predictions_{timestamp}.csv"
eval_df.to_csv(predictions_file, index=False)

print(f"📊 Results exported to: {results_file}")
print(f"📋 Detailed predictions saved to: {predictions_file}")

# Print summary
print("\n" + "="*60)
print("EVALUATION SUMMARY")
print("="*60)
print(f"Model: {model_config.model_name}")
print(f"Parameters: {total_params:,}")
print(f"Samples Evaluated: {len(eval_df)}")
print(f"\nPerformance Metrics:")
print(f"  MAE: {eval_df['abs_error'].mean():.4f}")
print(f"  RMSE: {np.sqrt((eval_df['error']**2).mean()):.4f}")
print(f"  Pearson Correlation: {eval_df['true_score'].corr(eval_df['predicted_score']):.4f}")
print(f"  Spearman Correlation: {eval_df['true_score'].corr(eval_df['predicted_score'], method='spearman'):.4f}")
print(f"  R²: {eval_df['true_score'].corr(eval_df['predicted_score'])**2:.4f}")
print("\n✅ Evaluation completed successfully!")
print("Next steps: Review results and consider model improvements.")